In [1]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity




In [2]:

# ------------------------------------------------------------
# ✅ 1. Test Data (paste your version here)
# ------------------------------------------------------------
test_data = {
    "Customers": [
        {"customer_id": 1001, "customer_name": "Fashion Store A", "country_id": 1},
        {"customer_id": 1002, "customer_name": "Sneaker Shop B", "country_id": 2},
        {"customer_id": 1003, "customer_name": "Urban Outfitters Plus", "country_id": 1},
        {"customer_id": 1004, "customer_name": "Premium Streetwear Co", "country_id": 3},
        {"customer_id": 1005, "customer_name": "Sporty Life Retail", "country_id": 2},
    ],

    "Countries": [
        {"country_id": 1, "country_name": "Netherlands"},
        {"country_id": 2, "country_name": "Germany"},
        {"country_id": 3, "country_name": "Belgium"}
    ],

    # ---- ITEMS & CLASSES ----------------------------------------------------
    "Items": [
        {"item_id": 1, "name": "T-Shirt Basic", "class_id": 10},
        {"item_id": 2, "name": "Jeans Slim Fit", "class_id": 10},
        {"item_id": 3, "name": "Sneakers White", "class_id": 20},
        {"item_id": 4, "name": "Hoodie Oversized", "class_id": 10},
        {"item_id": 5, "name": "Running Shoes Pro", "class_id": 20},
        {"item_id": 6, "name": "Baseball Cap", "class_id": 30},
        {"item_id": 7, "name": "Leather Jacket", "class_id": 40},
        {"item_id": 8, "name": "Winter Coat", "class_id": 40},
        {"item_id": 9, "name": "Tank Top", "class_id": 10},
        {"item_id": 10, "name": "Cargo Pants", "class_id": 10},
        {"item_id": 11, "name": "High-Top Sneakers", "class_id": 20},
        {"item_id": 12, "name": "Slip-On Shoes", "class_id": 20},
        {"item_id": 13, "name": "Graphic Tee", "class_id": 10},
        {"item_id": 14, "name": "Track Jacket", "class_id": 40},
        {"item_id": 15, "name": "Sport Shorts", "class_id": 30},
    ],

    "ItemClasses": [
        {"class_id": 10, "class_name": "Clothing"},
        {"class_id": 20, "class_name": "Shoes"},
        {"class_id": 30, "class_name": "Accessories"},
        {"class_id": 40, "class_name": "Outerwear"}
    ],

    "ItemClassValues": [
        {"value_id": 101, "class_id": 10, "value_name": "Tops"},
        {"value_id": 102, "class_id": 10, "value_name": "Bottoms"},
        {"value_id": 103, "class_id": 10, "value_name": "Sportswear"},
        {"value_id": 201, "class_id": 20, "value_name": "Sneakers"},
        {"value_id": 202, "class_id": 20, "value_name": "Running Shoes"},
        {"value_id": 301, "class_id": 30, "value_name": "Headwear"},
        {"value_id": 401, "class_id": 40, "value_name": "Jackets"},
        {"value_id": 402, "class_id": 40, "value_name": "Coats"},
    ],

    # ---- VARIANTS ------------------------------------------------------------
    "MatrixHeaders": [
        {"matrix_id": 1, "header_name": "Color"},
        {"matrix_id": 2, "header_name": "Size"}
    ],

    "MatrixParents": [
        {"parent_id": 1, "item_id": 1},
        {"parent_id": 2, "item_id": 3},
        {"parent_id": 3, "item_id": 4},
    ],

    "MatrixChildren": [
        {"child_id": 1, "parent_id": 1, "variant_value": "Red"},
        {"child_id": 2, "parent_id": 1, "variant_value": "Blue"},
        {"child_id": 3, "parent_id": 2, "variant_value": "White"},
        {"child_id": 4, "parent_id": 2, "variant_value": "Black"},
        {"child_id": 5, "parent_id": 3, "variant_value": "Green"},
        {"child_id": 6, "parent_id": 3, "variant_value": "Beige"},
    ],

    # ---- FAVORITES -----------------------------------------------------------
    "UserFavoriteItems": [
        {"user_id": 501, "item_id": 1},
        {"user_id": 501, "item_id": 3},
        {"user_id": 501, "item_id": 7},

        {"user_id": 502, "item_id": 2},
        {"user_id": 502, "item_id": 8},

        {"user_id": 503, "item_id": 5},
        {"user_id": 503, "item_id": 11},
        {"user_id": 503, "item_id": 15},

        {"user_id": 504, "item_id": 6},
        {"user_id": 504, "item_id": 13},

        {"user_id": 505, "item_id": 1},
        {"user_id": 505, "item_id": 14},
    ],

    # ---- USERS ---------------------------------------------------------------
    "Users": [
        {"user_id": 501, "name": "SalesRep1"},
        {"user_id": 502, "name": "SalesRep2"},
        {"user_id": 503, "name": "SalesRep3"},
        {"user_id": 504, "name": "SalesRep4"},
        {"user_id": 505, "name": "SalesRep5"},
    ],

    "SalesReps": [
        {"salesrep_id": 501, "customer_id": 1001},
        {"salesrep_id": 502, "customer_id": 1002},
        {"salesrep_id": 503, "customer_id": 1003},
        {"salesrep_id": 504, "customer_id": 1004},
        {"salesrep_id": 505, "customer_id": 1005},
    ],

    # ---- ORDERS --------------------------------------------------------------
    "Orders": [
        {"order_id": 9001, "customer_id": 1001},
        {"order_id": 9002, "customer_id": 1002},
        {"order_id": 9003, "customer_id": 1001},
        {"order_id": 9004, "customer_id": 1004},
        {"order_id": 9005, "customer_id": 1003},
        {"order_id": 9006, "customer_id": 1005},
        {"order_id": 9007, "customer_id": 1002},
        {"order_id": 9008, "customer_id": 1003}
    ],

    "OrderLines": [
        {"orderline_id": 1, "order_id": 9001, "item_id": 1, "quantity": 3},
        {"orderline_id": 2, "order_id": 9001, "item_id": 2, "quantity": 1},

        {"orderline_id": 3, "order_id": 9002, "item_id": 3, "quantity": 2},
        {"orderline_id": 4, "order_id": 9002, "item_id": 7, "quantity": 1},

        {"orderline_id": 5, "order_id": 9003, "item_id": 4, "quantity": 4},
        {"orderline_id": 6, "order_id": 9003, "item_id": 10, "quantity": 2},

        {"orderline_id": 7, "order_id": 9004, "item_id": 11, "quantity": 1},
        {"orderline_id": 8, "order_id": 9004, "item_id": 12, "quantity": 2},

        {"orderline_id": 9, "order_id": 9005, "item_id": 5, "quantity": 3},
        {"orderline_id": 10, "order_id": 9005, "item_id": 15, "quantity": 2},

        {"orderline_id": 11, "order_id": 9006, "item_id": 9, "quantity": 1},
        {"orderline_id": 12, "order_id": 9006, "item_id": 1, "quantity": 2},

        {"orderline_id": 13, "order_id": 9007, "item_id": 13, "quantity": 1},
        {"orderline_id": 14, "order_id": 9007, "item_id": 3, "quantity": 1},

        {"orderline_id": 15, "order_id": 9008, "item_id": 8, "quantity": 2},
        {"orderline_id": 16, "order_id": 9008, "item_id": 12, "quantity": 3},
    ]
}

In [3]:

# ------------------------------------------------------------
# ✅ 2. Convert entities to DataFrames
# ------------------------------------------------------------
items = pd.DataFrame(test_data["Items"])
classes = pd.DataFrame(test_data["ItemClasses"])
favorites = pd.DataFrame(test_data["UserFavoriteItems"])
order_lines = pd.DataFrame(test_data["OrderLines"])


In [4]:

# ------------------------------------------------------------
# ✅ 3. Build interaction data for Surprise
# Orders → strong signal (3)
# Favorites → medium signal (2)
# ------------------------------------------------------------
interactions = []

# From orders
for _, row in order_lines.iterrows():
    interactions.append([row["order_id"], row["item_id"], 3])

# From favorites
for _, row in favorites.iterrows():
    interactions.append([row["user_id"], row["item_id"], 2])

interactions_df = pd.DataFrame(interactions, columns=["user_id", "item_id", "rating"])


In [5]:
# ------------------------------------------------------------
# ✅ 4. Collaborative Filtering model (Surprise)
# ------------------------------------------------------------

reader = Reader(rating_scale=(1, 3))
data = Dataset.load_from_df(interactions_df, reader)

trainset = data.build_full_trainset()
cf_model = SVD()
cf_model.fit(trainset)



In [6]:
# ------------------------------------------------------------
# ✅ 5. Content-Based Filtering (TF-IDF on item metadata)
# ------------------------------------------------------------
items_full = items.merge(classes, on="class_id", how="left")

items_full["text"] = items_full["name"] + " " + items_full["class_name"]

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(items_full["text"])

content_sim = cosine_similarity(tfidf_matrix)



In [7]:


# ------------------------------------------------------------
# ✅ 6. Hybrid Scoring Function
# ------------------------------------------------------------
def hybrid_score(user_id, item_id):
    # Collaborative prediction
    cf_score = cf_model.predict(user_id, item_id).est / 3

    # Content-based similarity
    idx_item = items_full.index[items_full["item_id"] == item_id][0]

    favs = favorites[favorites["user_id"] == user_id]["item_id"].tolist()

    if favs:
        fav_indices = [
            items_full.index[items_full["item_id"] == f][0]
            for f in favs if f in items_full["item_id"].values
        ]
        content_score = content_sim[idx_item][fav_indices].mean()
    else:
        content_score = 0

    # Weighted hybrid
    return 0.6 * cf_score + 0.4 * content_score

In [8]:

# ------------------------------------------------------------
# ✅ 7. Generate Top‑N recommendations for a user
# ------------------------------------------------------------
def recommend_top_n(user_id, N=5):
    scores = []
    for item in items_full["item_id"]:
        score = hybrid_score(user_id, item)
        item_name = items_full.loc[items_full["item_id"] == item, "name"].values[0]

        scores.append((item, item_name, score))

    # Sort by highest score first
    scores.sort(key=lambda x: x[2], reverse=True)
    return scores[:N]


In [9]:
# ------------------------------------------------------------
# ✅ 8. Example Usage
# ------------------------------------------------------------
print("✅ Top 5 recommendations for user 501:")
print(recommend_top_n(501, 5))

print("\n✅ Top 5 recommendations for user 502:")
print(recommend_top_n(502, 5))


✅ Top 5 recommendations for user 501:
[(7, 'Leather Jacket', 0.603089832991269), (1, 'T-Shirt Basic', 0.6022988309695144), (3, 'Sneakers White', 0.5876673877420093), (14, 'Track Jacket', 0.5729355792644004), (12, 'Slip-On Shoes', 0.5628543691152974)]

✅ Top 5 recommendations for user 502:
[(2, 'Jeans Slim Fit', 0.6807241653191948), (8, 'Winter Coat', 0.6712721335153076), (9, 'Tank Top', 0.550403349790329), (1, 'T-Shirt Basic', 0.5492978315846718), (7, 'Leather Jacket', 0.5417851419257725)]
